# AERONET Multisite Availability and Filter-Day Overlap


In [1]:
NOTEBOOK_STEM = "04_aeronet_multisite_availability_overlap"


Objective: make the missing AERONET status table for all four sites: file availability,
date range, level/header fields, daily record count, matched filter-day count, and
whether this site can support a dust/column-context analysis.


In [2]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

REPO_ROOT = Path("/Users/ahmadjalil/github/aethmodular")
FTIR_DIR = REPO_ROOT / "research" / "ftir_hips_chem"
CATCH_UP_DIR = REPO_ROOT / "research" / "catch_up"
DATA_ROOT = Path('/Users/ahmadjalil/Library/CloudStorage/GoogleDrive-ahzs645@gmail.com/My Drive/University/Research/Grad/UC Davis Ann/NASA MAIA/Data')
OUT_DIR = CATCH_UP_DIR / "output" / globals().get("NOTEBOOK_STEM", Path.cwd().name)
OUT_DIR.mkdir(parents=True, exist_ok=True)

SCRIPTS_DIR = FTIR_DIR / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from config import SITES
from outliers import apply_exclusion_flags, apply_threshold_flags, get_clean_data
from plotting import PlotConfig, apply_default_style

apply_default_style()
PlotConfig.set(sites="all", layout="individual", show_stats=True, show_1to1=True)

SITE_CODES = {site: cfg["code"] for site, cfg in SITES.items()}
CODE_TO_SITE = {v: k for k, v in SITE_CODES.items()}
SITE_COLORS = {site: PlotConfig.get_site_color(site) for site in SITE_CODES}

PARAM_RENAME = {
    "EC_ftir": "ftir_ec",
    "OC_ftir": "ftir_oc",
    "HIPS_Fabs": "hips_fabs",
    "HIPS_T1": "hips_t1",
    "HIPS_R1": "hips_r1",
    "HIPS_t": "hips_t",
    "HIPS_r": "hips_r",
    "HIPS_tau": "hips_tau",
    "ChemSpec_EC_PM2.5": "chemspec_ec",
    "ChemSpec_OC_PM2.5": "chemspec_oc",
    "ChemSpec_OM_PM2.5": "chemspec_om",
    "ChemSpec_Iron_PM2.5": "iron",
    "ChemSpec_Silicon_PM2.5": "silicon",
    "ChemSpec_Aluminum_PM2.5": "aluminum",
    "ChemSpec_Calcium_PM2.5": "calcium",
    "ChemSpec_Titanium_PM2.5": "titanium",
    "ChemSpec_Filter_PM2.5_mass": "pm25_mass",
}

def _first_existing(paths):
    for p in paths:
        if Path(p).exists():
            return Path(p)
    raise FileNotFoundError("None of these paths exist:\n" + "\n".join(map(str, paths)))

def load_filter_long():
    path = _first_existing([
        FTIR_DIR / "Filter Data" / "unified_filter_dataset.pkl",
        DATA_ROOT / "Combine csv files" / "Filter Data" / "unified_filter_dataset.pkl",
        DATA_ROOT / "Combine csv files" / "FTIR_HIPS_Chem" / "Filter Data" / "unified_filter_dataset.pkl",
    ])
    df = pd.read_pickle(path)
    df["SampleDate"] = pd.to_datetime(df["SampleDate"])
    df["base_filter_id"] = df["FilterId"].astype(str).str.replace(r"-\d+$", "", regex=True)
    print(f"Loaded filter data: {path}  rows={len(df):,}")
    return df

def load_filter_wide(params):
    long = load_filter_long()
    d = long[long["Parameter"].isin(params)].copy()
    meta = (
        d.sort_values(["Site", "base_filter_id", "SampleDate"])
         .groupby(["Site", "base_filter_id"], as_index=False)
         .agg(
             filter_id=("FilterId", "first"),
             date=("SampleDate", "first"),
             volume_m3=("Volume_m3", "max"),
             deposit_area_cm2=("DepositArea_cm2", "max"),
             lot_id=("LotId", "first"),
         )
    )
    conc = d.pivot_table(
        index=["Site", "base_filter_id"],
        columns="Parameter",
        values="Concentration",
        aggfunc="first",
    ).rename(columns=PARAM_RENAME)
    mass = d.pivot_table(
        index=["Site", "base_filter_id"],
        columns="Parameter",
        values="MassLoading_ug",
        aggfunc="first",
    ).rename(columns={p: PARAM_RENAME.get(p, p) + "_mass_ug" for p in params})
    wide = meta.merge(conc.reset_index(), on=["Site", "base_filter_id"], how="left")
    wide = wide.merge(mass.reset_index(), on=["Site", "base_filter_id"], how="left")
    wide["site"] = wide["Site"].map(CODE_TO_SITE)
    return wide

def load_aeth_site(site):
    file_map = {
        "Beijing": "df_Beijing_9am_resampled.pkl",
        "Delhi": "df_Delhi_9am_resampled.pkl",
        "JPL": "df_JPL_9am_resampled.pkl",
        "Addis_Ababa": "df_Addis_Ababa_9am_resampled.pkl",
    }
    repo_path = FTIR_DIR / "processed_sites" / file_map[site]
    cloud_candidates = [
        DATA_ROOT / "Aethelometry Data" / "JacrosMA350 60s Data20250804082112" / "df_Jacros_9am_resampled.pkl",
        DATA_ROOT / "Aethelometry Data" / "Kyan Data" / "Dataset" / "df_cleaned_Beijing_manual_BCc.pkl",
        DATA_ROOT / "Aethelometry Data" / "Kyan Data" / "Dataset" / "df_cleaned_Delhi_manual_BCc.pkl",
        DATA_ROOT / "Aethelometry Data" / "Kyan Data" / "Dataset" / "df_cleaned_JPL_manual_BCc.pkl",
    ]
    path = repo_path if repo_path.exists() else _first_existing([p for p in cloud_candidates if site.replace("_Ababa", "") in p.name or site == "Addis_Ababa"])
    df = pd.read_pickle(path)
    df = df.loc[:, ~df.columns.duplicated()].copy()
    if "day_9am" in df.columns:
        df["date"] = pd.to_datetime(df["day_9am"]).dt.normalize()
    elif "datetime_local" in df.columns:
        df["date"] = pd.to_datetime(df["datetime_local"]).dt.normalize()
    else:
        df["date"] = pd.to_datetime(df.index).normalize()
    df["site"] = site
    return df

def _to_ugm3(s):
    s = pd.to_numeric(s, errors="coerce")
    med = s.dropna().abs().median()
    return s / 1000.0 if pd.notna(med) and med > 100 else s

def aeth_metrics(site):
    df = load_aeth_site(site)
    out = df[["site", "date"]].copy()
    for wl in ["UV", "Blue", "Green", "Red", "IR"]:
        col = f"{wl} BCc"
        out[f"{wl.lower()}_bc_ugm3"] = _to_ugm3(df[col]) if col in df.columns else np.nan
    out["aeth_ir_ugm3"] = out["ir_bc_ugm3"]
    out["uv_ir_bcc_ratio"] = out["uv_bc_ugm3"] / out["ir_bc_ugm3"]
    out["green_ir_bcc_ratio"] = out["green_bc_ugm3"] / out["ir_bc_ugm3"]
    out["delta_c_ugm3"] = out["uv_bc_ugm3"] - out["ir_bc_ugm3"]
    # Raw attenuation-based apparent AAE from rolling mean dATN when available.
    uv = pd.to_numeric(df.get("delta UV ATN1 rolling mean", np.nan), errors="coerce")
    ir = pd.to_numeric(df.get("delta IR ATN1 rolling mean", np.nan), errors="coerce")
    mask = (uv > 0) & (ir > 0)
    out["aae_atn_uv_ir"] = np.where(mask, -np.log(uv / ir) / np.log(375 / 880), np.nan)
    return out.groupby(["site", "date"], as_index=False).median(numeric_only=True)

def matched_dataset(include_chem=True):
    params = ["EC_ftir", "OC_ftir", "HIPS_Fabs", "HIPS_T1", "HIPS_R1", "HIPS_t", "HIPS_r", "HIPS_tau"]
    if include_chem:
        params += [
            "ChemSpec_EC_PM2.5", "ChemSpec_OC_PM2.5", "ChemSpec_OM_PM2.5",
            "ChemSpec_Iron_PM2.5", "ChemSpec_Silicon_PM2.5", "ChemSpec_Aluminum_PM2.5",
            "ChemSpec_Calcium_PM2.5", "ChemSpec_Titanium_PM2.5", "ChemSpec_Filter_PM2.5_mass",
        ]
    filt = load_filter_wide(params)
    filt["date"] = pd.to_datetime(filt["date"]).dt.normalize()
    aeth = pd.concat([aeth_metrics(s) for s in SITE_CODES], ignore_index=True)
    m = filt.merge(aeth, on=["site", "date"], how="left")
    # Units: ChemSpec metals are usually ng/m3; convert common tracers to ug/m3 for ratios.
    for col in ["iron", "silicon", "aluminum", "calcium", "titanium"]:
        if col in m.columns:
            med = pd.to_numeric(m[col], errors="coerce").dropna().abs().median()
            if pd.notna(med) and med > 50:
                m[col + "_ugm3"] = m[col] / 1000.0
            else:
                m[col + "_ugm3"] = m[col]
    m["ec_mass_ug"] = m.get("ftir_ec_mass_ug")
    if "ftir_ec" in m.columns:
        m["ec_mass_from_volume_ug"] = m["ftir_ec"] * m["volume_m3"]
        m["ec_mass_ug"] = m["ec_mass_ug"].fillna(m["ec_mass_from_volume_ug"])
    m["ec_surface_loading_ug_cm2"] = m["ec_mass_ug"] / m["deposit_area_cm2"]
    m["hips_bc_mac10_ugm3"] = m["hips_fabs"] / 10.0
    m["hips_minus_ftir"] = m["hips_bc_mac10_ugm3"] - m["ftir_ec"]
    m["hips_to_ftir_ratio"] = m["hips_bc_mac10_ugm3"] / m["ftir_ec"]
    m["oc_ec_ratio"] = m["ftir_oc"] / m["ftir_ec"]
    return m

def add_project_exclusion_flags(df):
    # Add project exclusion/outlier flags using research/ftir_hips_chem/scripts/outliers.py.
    parts = []
    for site in SITE_CODES:
        site_df = df[df["site"] == site].copy()
        if site_df.empty:
            continue
        flag_input = site_df.copy()
        flag_input["filter_id"] = flag_input["base_filter_id"]
        flag_input["aeth_bc"] = flag_input["aeth_ir_ugm3"] * 1000.0
        flag_input["filter_ec"] = flag_input["ftir_ec"] * 1000.0
        flag_input = apply_exclusion_flags(flag_input, site)
        flag_input = apply_threshold_flags(flag_input, site)
        site_df["is_excluded"] = flag_input["is_excluded"].values
        site_df["exclusion_reason"] = flag_input.get("exclusion_reason", "").values
        site_df["is_outlier"] = flag_input["is_outlier"].values
        site_df["outlier_reason"] = flag_input.get("outlier_reason", "").values
        parts.append(site_df)
    out = pd.concat(parts, ignore_index=True) if parts else df.copy()
    out["is_clean"] = ~(out.get("is_excluded", False) | out.get("is_outlier", False))
    return out

def regression_row(df, x, y, label):
    d = df[[x, y]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(d) < 3:
        return {"label": label, "x": x, "y": y, "n": len(d), "slope": np.nan, "intercept": np.nan, "r2": np.nan, "p": np.nan}
    lr = stats.linregress(d[x], d[y])
    return {"label": label, "x": x, "y": y, "n": len(d), "slope": lr.slope, "intercept": lr.intercept, "r2": lr.rvalue**2, "p": lr.pvalue}

def save_table(df, name):
    path = OUT_DIR / name
    df.to_csv(path, index=False)
    print(f"Wrote {path}")
    return path


In [3]:
def read_aeronet_csv(path):
    path = Path(path)
    lines = path.read_text(errors="ignore").splitlines()
    header_idx = None
    for i, line in enumerate(lines):
        if "Date(" in line and "," in line:
            header_idx = i
            break
    if header_idx is None:
        return None
    df = pd.read_csv(path, skiprows=header_idx)
    date_col = next((c for c in df.columns if c.startswith("Date(")), None)
    time_col = next((c for c in df.columns if c.startswith("Time(")), None)
    if date_col:
        if time_col:
            dt = df[date_col].astype(str) + " " + df[time_col].astype(str)
            df["datetime"] = pd.to_datetime(dt, errors="coerce", dayfirst=True)
        else:
            df["datetime"] = pd.to_datetime(df[date_col], errors="coerce", dayfirst=True)
        df["date"] = df["datetime"].dt.normalize()
    df = df.replace(-999, np.nan)
    return df

def aeronet_candidates(site):
    repo = {
        "Beijing": [REPO_ROOT / "notebooks/analysis/data/aeronet_aod_beijing.csv"],
        "Delhi": [REPO_ROOT / "notebooks/analysis/data/aeronet_aod_delhi.csv"],
        "JPL": [REPO_ROOT / "notebooks/analysis/data/aeronet_aod_jpl.csv"],
        "Addis_Ababa": list((DATA_ROOT / "AERONET" / "Jacros").glob("**/*.csv")),
    }
    return [p for p in repo.get(site, []) if Path(p).exists()]

filters = load_filter_wide(["EC_ftir", "HIPS_Fabs"])
filter_days = filters.groupby("site")["date"].apply(lambda s: set(pd.to_datetime(s).dt.normalize().dropna())).to_dict()

rows = []
for site in SITE_CODES:
    paths = aeronet_candidates(site)
    if not paths:
        rows.append({"site": site, "file": None, "status": "missing_file"})
        continue
    for path in paths:
        df = read_aeronet_csv(path)
        if df is None or "date" not in df:
            rows.append({"site": site, "file": str(path), "status": "unreadable_or_no_date"})
            continue
        days = set(df["date"].dropna())
        overlap = days.intersection(filter_days.get(site, set()))
        aod_cols = [c for c in df.columns if "AOD" in c.upper()]
        fine_cols = [c for c in df.columns if "FINE" in c.upper() or "FMF" in c.upper()]
        coarse_cols = [c for c in df.columns if "COARSE" in c.upper()]
        rows.append({
            "site": site,
            "file": str(path),
            "status": "ok",
            "aeronet_rows": len(df),
            "aeronet_days": len(days),
            "aeronet_start": min(days) if days else pd.NaT,
            "aeronet_end": max(days) if days else pd.NaT,
            "filter_days": len(filter_days.get(site, set())),
            "matched_filter_days": len(overlap),
            "n_aod_columns": len(aod_cols),
            "n_fine_columns": len(fine_cols),
            "n_coarse_columns": len(coarse_cols),
            "example_aod_columns": "; ".join(aod_cols[:5]),
            "example_fine_columns": "; ".join(fine_cols[:5]),
            "example_coarse_columns": "; ".join(coarse_cols[:5]),
        })
availability = pd.DataFrame(rows)
save_table(availability, "aeronet_multisite_availability_overlap.csv")
availability


Loaded filter data: /Users/ahmadjalil/github/aethmodular/research/ftir_hips_chem/Filter Data/unified_filter_dataset.pkl  rows=44,493
Wrote /Users/ahmadjalil/github/aethmodular/research/catch_up/output/04_aeronet_multisite_availability_overlap/aeronet_multisite_availability_overlap.csv


/var/folders/7k/65ckdzsj0w3171qv80rh8dgr0000gn/T/ipykernel_91059/669371373.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["datetime"] = pd.to_datetime(dt, errors="coerce", dayfirst=True)
/var/folders/7k/65ckdzsj0w3171qv80rh8dgr0000gn/T/ipykernel_91059/669371373.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["datetime"] = pd.to_datetime(dt, errors="coerce", dayfirst=True)
/var/folders/7k/65ckdzsj0w3171qv80rh8dgr0000gn/T/ipykernel_91059/669371373.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["datetime"] = pd.to_datetime(dt, errors="coerce", dayfirs

,site,file,status,aeronet_rows,aeronet_days,aeronet_start,aeronet_end,filter_days,matched_filter_days,n_aod_columns,n_fine_columns,n_coarse_columns,example_aod_columns,example_fine_columns,example_coarse_columns
0,Beijing,/Users/ahmadjalil/github/aethmodular/notebooks...,ok,346.0,1.0,2026-05-13,2026-05-13,190.0,0.0,58.0,0.0,0.0,AOD_1640nm; AOD_1020nm; AOD_870nm; AOD_865nm; ...,,
1,Delhi,/Users/ahmadjalil/github/aethmodular/notebooks...,ok,659.0,1.0,2026-05-13,2026-05-13,97.0,0.0,58.0,0.0,0.0,AOD_1640nm; AOD_1020nm; AOD_870nm; AOD_865nm; ...,,
2,JPL,/Users/ahmadjalil/github/aethmodular/notebooks...,ok,989.0,1.0,2026-05-13,2026-05-13,158.0,0.0,58.0,0.0,0.0,AOD_1640nm; AOD_1020nm; AOD_870nm; AOD_865nm; ...,,
3,Addis_Ababa,None,missing_file,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
